In [1]:
%pip install webdriver-manager

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pandas


Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [4]:
# imports clave para el funcionamiento de todo el scrapping

from selenium import webdriver
from selenium. webdriver.chrome.options import Options
from selenium. webdriver.common.by import By
import pandas as pd
import time
import re

In [5]:
#imports necesarios para pasar página y seguir escrapeando más de 10 artículos

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [6]:
#Agregado para probar una mejor versión del scrapping

import requests
import json

In [7]:
# ==========================
# CONFIGURACIÓN
# ==========================
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-notifications")

driver = webdriver.Chrome(options=options)

url = "https://www.elespectador.com/buscador/migraci%C3%B3n-venezolana/"
driver.get(url)

print("⏳ Esperando carga inicial...")
time.sleep(10)

⏳ Esperando carga inicial...


In [8]:
# ============================================
# FUNCIÓN PARA LA EXTRACCIÓN DE LOS ARTÍCULOS
# ============================================


def extraction():

    print("\n" + "=" * 60)
    print("📊 EXTRAYENDO DATOS DESDE API searcherTag")
    print("=" * 60)

    titulares = []
    links = []
    categorias = []
    cuerpos = []

    api_url = "https://www.elespectador.com/pf/api/v3/content/fetch/searcherTag"

    # ==========================
    # OBTENER TODAS LAS NOTICIAS
    # ==========================

    total_noticias = None

    for offset in range(0, 20, 10):

        print(f"\n📄 Descargando bloque desde {offset}")

        query = {
            "author": None,
            "date": None,
            "from": offset,
            "keyword": "migración-venezolana",
            "section": None,
            "size": 10,
            "subtype": None
        }

        params = {
            "query": json.dumps(query, ensure_ascii=False),
            "d": "1198",
            "mxId": "00000000",
            "_website": "el-espectador"
        }

        try:

            response = requests.get(
                api_url,
                params=params,
                timeout=30
            )

            response.raise_for_status()

            data = response.json()

            # Obtener total de resultados sólo una vez
            if total_noticias is None:
                total_noticias = data.get("count", 0)

                print(
                    f"✅ Total de noticias encontradas: "
                    f"{total_noticias}"
                )

            noticias = data.get("content_elements", [])

            if len(noticias) == 0:
                print("⚠️ No se encontraron más noticias")
                break

            for noticia in noticias:

                try:

                    titulo = noticia["headlines"]["basic"]

                    categoria = noticia["taxonomy"][
                        "primary_section"
                    ]["name"]

                    url = (
                        "https://www.elespectador.com"
                        + noticia["canonical_url"]
                    )

                    titulares.append(titulo)
                    categorias.append(categoria)
                    links.append(url)

                except Exception as e:
                    print(f"⚠️ Error procesando noticia: {e}")

            # Si ya llegamos al total, terminar
            if total_noticias and len(titulares) >= total_noticias:
                break

        except Exception as e:
            print(f"❌ Error API: {e}")
            break

    print("\n" + "=" * 60)
    print("✅ EXTRACCIÓN DE METADATOS COMPLETADA")
    print("=" * 60)

    print(f"Títulos encontrados: {len(titulares)}")
    print(f"Links encontrados: {len(links)}")

    # ==========================
    # EXTRAER CUERPO ARTÍCULOS
    # ==========================

    print("\n" + "=" * 60)
    print("📰 EXTRAYENDO CUERPOS DE ARTÍCULOS")
    print("=" * 60)

    for i, url in enumerate(links, start=1):

        try:

            print(f"[{i}/{len(links)}] {url}")

            

            driver.get(url)

            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located(
                    (By.TAG_NAME, "article")
                )
            )

            parrafos = driver.find_elements(
                By.CSS_SELECTOR,
                "article p"
            )

            if not parrafos:

                parrafos = driver.find_elements(
                    By.CSS_SELECTOR,
                    '[class*="article"] p'
                )

            if not parrafos:

                parrafos = driver.find_elements(
                    By.TAG_NAME,
                    "p"
                )

            texto_articulo = " ".join(

                p.text.strip()

                for p in parrafos

                if len(p.text.strip()) > 30

            )

            cuerpos.append(texto_articulo)

        except Exception as e:

            print(f"❌ Error en {url}: {e}")

            cuerpos.append("")

    # ==========================
    # DATAFRAME FINAL
    # ==========================

    df = pd.DataFrame({

        "Titulo": titulares,

        "Categoria": categorias,

        "URL": links,

        "Cuerpo": cuerpos

    })

    print("\n" + "=" * 60)
    print("📈 RESUMEN FINAL")
    print("=" * 60)

    print(f"Noticias extraídas: {len(df)}")

    df.to_csv(
        "migracion_venezolana.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("✅ Archivo guardado: migracion_venezolana.csv")

    return df

In [9]:
extraction()



📊 EXTRAYENDO DATOS DESDE API searcherTag

📄 Descargando bloque desde 0
✅ Total de noticias encontradas: 235

📄 Descargando bloque desde 10

✅ EXTRACCIÓN DE METADATOS COMPLETADA
Títulos encontrados: 20
Links encontrados: 20

📰 EXTRAYENDO CUERPOS DE ARTÍCULOS
[1/20] https://www.elespectador.com/mundo/venezuela/migrantes-en-bello-antioquia-asi-deben-actualizar-sus-datos-para-no-perder-el-servicio-de-salud/
[2/20] https://www.elespectador.com/mundo/america/petro-en-ee-uu-hay-que-avanzar-mas-en-los-derechos-de-los-migrantes-venezolanos/
[3/20] https://www.elespectador.com/el-magazin-cultural/emigro-luego-existo-la-diaspora-venezolana-narrada-desde-el-grafiti/
[4/20] https://www.elespectador.com/el-magazin-cultural/hemos-llegado-a-berlin-la-migracion-venezolana-vista-desde-la-infancia/
[5/20] https://www.elespectador.com/colombia/mas-regiones/lo-que-fue-y-lo-que-ahora-es-la-migracion-de-venezolanos-a-cucuta/
[6/20] https://www.elespectador.com/colombia/mas-regiones/el-desafio-migratorio-no-

,Titulo,Categoria,URL,Cuerpo
0,Migrantes en Bello: así deben actualizar sus d...,Venezuela,https://www.elespectador.com/mundo/venezuela/m...,Audio generado con IA de Google La Alcaldía de...
1,Petro en EE. UU.: “Hay que avanzar más” en los...,América,https://www.elespectador.com/mundo/america/pet...,Audio generado con IA de Google En el marco de...
2,"“Emigro, luego existo”: la diáspora venezolana...",El Magazín Cultural,https://www.elespectador.com/el-magazin-cultur...,"En la calle 51 con carrera 7, un mural amarill..."
3,“Hemos llegado a Berlín”: la migración venezol...,El Magazín Cultural,https://www.elespectador.com/el-magazin-cultur...,"Entre la incertidumbre y la resistencia, la ex..."
4,Los cambios en la migración de venezolanos a C...,Más regiones,https://www.elespectador.com/colombia/mas-regi...,Audio generado con IA de Google Hace ocho años...
5,"El desafío migratorio no cesa en Maicao, La Gu...",Más regiones,https://www.elespectador.com/colombia/mas-regi...,"El municipio de Maicao, en el norte de La Guaj..."
6,"Trump, Bukele y migrantes venezolanos: una com...",América,https://www.elespectador.com/mundo/america/tru...,“Él es un buen muchacho y lo puedo demostrar d...
7,La migración venezolana cambia de ruta,Venezuela,https://www.elespectador.com/mundo/venezuela/l...,Audio generado con IA de Google Lejos de deten...
8,Gobierno alista cirugía a solicitudes de asilo...,Política,https://www.elespectador.com/politica/asilo-en...,La administración del presidente Gustavo Petro...
9,"A orillas de Venezuela, los autores que viven ...",El Magazín Cultural,https://www.elespectador.com/el-magazin-cultur...,Audio generado con IA de Google La problemátic...
